# Assignment 2: ML Model Training & Evaluation

## Part 0: Data Cleaning

### Step 1: Data Ingestion and Storage

In [1]:
from urllib.request import urlretrieve
from pathlib import Path

raw_dir = Path("data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

files = [
    ("https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet", raw_dir/"yellow_taxi_data.parquet"),
    ("https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv", raw_dir/"taxi_zone_lookup.csv"),
]

for url, filename in files:
    urlretrieve(url, filename)

print("Done!")

Done!


In [2]:
import polars as pl

df = pl.read_parquet(raw_dir/'yellow_taxi_data.parquet')
df.head()

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
i32,datetime[ns],datetime[ns],i64,f64,i64,str,i32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2,2024-01-01 00:57:55,2024-01-01 01:17:43,1,1.72,1,"""N""",186,79,2,17.7,1.0,0.5,0.0,0.0,1.0,22.7,2.5,0.0
1,2024-01-01 00:03:00,2024-01-01 00:09:36,1,1.8,1,"""N""",140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
1,2024-01-01 00:17:06,2024-01-01 00:35:01,1,4.7,1,"""N""",236,79,1,23.3,3.5,0.5,3.0,0.0,1.0,31.3,2.5,0.0
1,2024-01-01 00:36:38,2024-01-01 00:44:56,1,1.4,1,"""N""",79,211,1,10.0,3.5,0.5,2.0,0.0,1.0,17.0,2.5,0.0
1,2024-01-01 00:46:51,2024-01-01 00:52:57,1,0.8,1,"""N""",211,148,1,7.9,3.5,0.5,3.2,0.0,1.0,16.1,2.5,0.0


In [3]:
import sys
expected_result = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "PULocationID", "DOLocationID", "passenger_count", "trip_distance", "fare_amount", "tip_amount", "total_amount", "payment_type"]

try:
    missing = [m for m in expected_result if m not in df.columns]
    if missing:
        raise ValueError(f"Missing expected columns: {missing}")
    print("All columns present")

    if df.schema.get("tpep_pickup_datetime") != pl.Datetime:
        raise TypeError(f"tpep_pickup_datetime is {df.schema.get('tpep_pickup_datetime')}, expected Datetime")
    print("tpep_pickup_datetime datatype is: ", df.schema.get("tpep_pickup_datetime"))

    if df.schema.get("tpep_dropoff_datetime") != pl.Datetime:
        raise TypeError(f"tpep_dropoff_datetime is {df.schema.get('tpep_dropoff_datetime')}, expected Datetime")
    print("tpep_dropoff_datetime datatype is: ", df.schema.get("tpep_dropoff_datetime"))

    print(f'Number of rows: {len(df):,}')
    print(f'Number of columns: {len(df.columns)}')
    print('\nColumn names and types:')
    print(df.schema)

except Exception as e:
    print(f"Data Validation failed: {e}")
    sys.exit(1)


All columns present
tpep_pickup_datetime datatype is:  Datetime(time_unit='ns', time_zone=None)
tpep_dropoff_datetime datatype is:  Datetime(time_unit='ns', time_zone=None)
Number of rows: 2,964,624
Number of columns: 19

Column names and types:
Schema({'VendorID': Int32, 'tpep_pickup_datetime': Datetime(time_unit='ns', time_zone=None), 'tpep_dropoff_datetime': Datetime(time_unit='ns', time_zone=None), 'passenger_count': Int64, 'trip_distance': Float64, 'RatecodeID': Int64, 'store_and_fwd_flag': String, 'PULocationID': Int32, 'DOLocationID': Int32, 'payment_type': Int64, 'fare_amount': Float64, 'extra': Float64, 'mta_tax': Float64, 'tip_amount': Float64, 'tolls_amount': Float64, 'improvement_surcharge': Float64, 'total_amount': Float64, 'congestion_surcharge': Float64, 'Airport_fee': Float64})


### Step 2: Data Transformation & Analysis

In [4]:
# ensuring the data is clean to avoid issues in analysis and visualizations
# removing nulls values in all columns
df_clean = df.drop_nulls()
print(f'Number of rows after dropping nulls: {len(df_clean):,}')
print(f'Number of rows removed after dropping nulls: {len(df) - len(df_clean):,}')

#filtering invalid trips: trips with zero or negative distance, negative fares, or fares exceeding $500
df_invalid = (
    df_clean
    .filter((pl.col("fare_amount") > 0) & (pl.col("fare_amount") <= 500))
    .filter(pl.col("trip_distance") > 0)
)
print(f'Number of rows after filtering invalid trips: {len(df_invalid):,}')
print(f'Number of rows removed after filtering invalid trips: {len(df_clean) - len(df_invalid):,}')

#removing trips where dropoff time is before pickup time
df_final = df_invalid.filter(pl.col("tpep_dropoff_datetime") >= pl.col("tpep_pickup_datetime"))
print(f'Number of rows after filtering dropoff before pickup: {len(df_final):,}')
print(f'Number of rows removed after filtering dropoff before pickup: {len(df_invalid) - len(df_final):,}')


Number of rows after dropping nulls: 2,824,462
Number of rows removed after dropping nulls: 140,162
Number of rows after filtering invalid trips: 2,754,435
Number of rows removed after filtering invalid trips: 70,027
Number of rows after filtering dropoff before pickup: 2,754,427
Number of rows removed after filtering dropoff before pickup: 8


In [5]:
#adding new column for trip duration in minutes
enriched = df_final.with_columns([
    ((pl.col("tpep_dropoff_datetime") - pl.col("tpep_pickup_datetime"))
    .dt.total_seconds() / 60).alias('trip_duration_minutes'),
])

enriched.select(pl.col("trip_duration_minutes")).head()

trip_duration_minutes
f64
19.8
6.6
17.916667
8.3
6.1


In [6]:
#adding new column for trip speed
enriched = enriched.with_columns([
    (
        pl.when(pl.col("trip_duration_minutes") > 0)
        .then(pl.col("trip_distance") / (pl.col("trip_duration_minutes") / 60))
        .otherwise(None)
        .alias('trip_speed_mph')
    )
])

enriched.select(pl.col("trip_speed_mph"))

trip_speed_mph
f64
5.212121
16.363636
15.739535
10.120482
7.868852
…
26.215768
12.205853
11.797418


In [7]:
#adding new columns for pickup hour and weekday
enriched = enriched.with_columns([
    pl.col('tpep_pickup_datetime').dt.hour().alias('pickup_hour'),
    pl.col('tpep_pickup_datetime').dt.weekday().alias('pickup_weekday')
])

enriched.select(pl.col("pickup_hour"), pl.col("pickup_weekday")).head()

pickup_hour,pickup_weekday
i8,i8
0,1
0,1
0,1
0,1
0,1


In [8]:
print(enriched.schema)

Schema({'VendorID': Int32, 'tpep_pickup_datetime': Datetime(time_unit='ns', time_zone=None), 'tpep_dropoff_datetime': Datetime(time_unit='ns', time_zone=None), 'passenger_count': Int64, 'trip_distance': Float64, 'RatecodeID': Int64, 'store_and_fwd_flag': String, 'PULocationID': Int32, 'DOLocationID': Int32, 'payment_type': Int64, 'fare_amount': Float64, 'extra': Float64, 'mta_tax': Float64, 'tip_amount': Float64, 'tolls_amount': Float64, 'improvement_surcharge': Float64, 'total_amount': Float64, 'congestion_surcharge': Float64, 'Airport_fee': Float64, 'trip_duration_minutes': Float64, 'trip_speed_mph': Float64, 'pickup_hour': Int8, 'pickup_weekday': Int8})


In [9]:
print(enriched["payment_type"])

shape: (2_754_427,)
Series: 'payment_type' [i64]
[
	2
	1
	1
	1
	1
	…
	1
	1
	1
	2
	1
]


In [10]:
# filtering the dataset to include only credit card transactions for the modeling tasks
enriched = enriched.filter(pl.col("payment_type").is_in([1]))


In [11]:
print(enriched["payment_type"])

shape: (2_298_380,)
Series: 'payment_type' [i64]
[
	1
	1
	1
	1
	1
	…
	1
	1
	1
	1
	1
]


In [12]:
print(len(enriched))    #dropped from 2.9 mil to 2.2 mil after filtering for credit card transactions

2298380


In [13]:
enriched.write_parquet(raw_dir/'yellow_taxi_data_enriched.parquet')

## Part 1: Data Preprocessing & Feature Engineering

### 1. Feature Engineering

In [14]:
#temporal features

enriched = enriched.with_columns([
    pl.col('tpep_pickup_datetime').dt.hour().alias('pickup_hour'),
    (pl.col('tpep_pickup_datetime').dt.weekday() - 1).alias('pickup_day_of_week')
])

#days of the week 0 - 6 so weekend is 5 and 6
enriched = enriched.with_columns(
    (pl.col('pickup_day_of_week') >= 5).alias('is_weekend')
)

enriched.select(["tpep_pickup_datetime","pickup_hour","pickup_day_of_week","is_weekend"]).sample(n=5)

tpep_pickup_datetime,pickup_hour,pickup_day_of_week,is_weekend
datetime[ns],i8,i8,bool
2024-01-24 16:32:32,16,2,false
2024-01-06 12:51:03,12,5,true
2024-01-20 09:40:17,9,5,true
2024-01-19 20:09:03,20,4,false
2024-01-05 15:13:50,15,4,false


In [15]:
#trip features

#trip duration minutes
enriched = enriched.with_columns([
    ((pl.col("tpep_dropoff_datetime") - pl.col("tpep_pickup_datetime"))
    .dt.total_seconds() / 60).alias('trip_duration_minutes')
])

#trip speed mph
enriched = enriched.with_columns([
    (
        pl.when(pl.col("trip_duration_minutes") > 0)
        .then(pl.col("trip_distance") / (pl.col("trip_duration_minutes") / 60))
        .otherwise(None)
        .alias('trip_speed_mph')
    )
])

#log trip distance
enriched = enriched.with_columns([
    pl.col("trip_distance").log1p().alias("log_trip_distance")
])

enriched.sample(n=5)

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,trip_duration_minutes,trip_speed_mph,pickup_hour,pickup_weekday,pickup_day_of_week,is_weekend,log_trip_distance
i32,datetime[ns],datetime[ns],i64,f64,i64,str,i32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8,i8,i8,bool,f64
2,2024-01-14 15:45:52,2024-01-14 15:55:03,1,1.18,1,"""N""",113,158,1,10.0,0.0,0.5,2.8,0.0,1.0,16.8,2.5,0.0,9.183333,7.709619,15,7,6,true,0.779325
2,2024-01-21 00:03:46,2024-01-21 00:23:51,2,4.94,1,"""N""",113,239,1,24.7,1.0,0.5,5.94,0.0,1.0,35.64,2.5,0.0,20.083333,14.758506,0,7,6,true,1.781709
1,2024-01-24 15:12:23,2024-01-24 15:21:24,1,0.8,99,"""N""",42,42,1,15.5,0.0,0.5,0.0,0.0,1.0,17.0,0.0,0.0,9.016667,5.323475,15,3,2,false,0.587787
2,2024-01-13 13:00:22,2024-01-13 13:09:16,1,2.99,1,"""N""",87,66,1,14.9,0.0,0.5,3.78,0.0,1.0,22.68,2.5,0.0,8.9,20.157303,13,6,5,true,1.383791
2,2024-01-04 15:58:12,2024-01-04 16:11:10,1,1.89,1,"""N""",140,161,1,13.5,0.0,0.5,1.75,0.0,1.0,19.25,2.5,0.0,12.966667,8.745501,15,4,3,false,1.061257


In [16]:
# fare features
#fare_per_mile
enriched = enriched.with_columns([
    (
        pl.when(pl.col("trip_distance") > 0)
        .then (pl.col("fare_amount") / pl.col("trip_distance"))
        .otherwise(None)
        .alias('fare_per_mile')
    )
])

#fare_per_minute
enriched = enriched.with_columns([
    (
        pl.when(pl.col("trip_duration_minutes") > 0)
        .then(pl.col("fare_amount") / pl.col("trip_duration_minutes"))
        .otherwise(None)
        .alias('fare_per_minute')
    )
])

enriched.sample(n=5)

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,trip_duration_minutes,trip_speed_mph,pickup_hour,pickup_weekday,pickup_day_of_week,is_weekend,log_trip_distance,fare_per_mile,fare_per_minute
i32,datetime[ns],datetime[ns],i64,f64,i64,str,i32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8,i8,i8,bool,f64,f64,f64
2,2024-01-04 23:59:28,2024-01-05 00:32:26,1,17.6,2,"""N""",132,90,1,70.0,0.0,0.5,16.19,6.94,1.0,98.88,2.5,1.75,32.966667,32.032356,23,4,3,false,2.923162,3.977273,2.123357
1,2024-01-27 21:56:18,2024-01-27 21:59:48,2,0.5,1,"""N""",161,163,1,5.8,3.5,0.5,5.0,0.0,1.0,15.8,2.5,0.0,3.5,8.571429,21,6,5,true,0.405465,11.6,1.657143
2,2024-01-17 14:50:59,2024-01-17 15:21:04,1,2.33,1,"""N""",234,50,1,24.7,0.0,0.5,5.74,0.0,1.0,34.44,2.5,0.0,30.083333,4.647091,14,3,2,false,1.202972,10.600858,0.821053
1,2024-01-01 22:56:32,2024-01-01 23:18:24,1,9.0,1,"""N""",138,48,1,37.3,10.25,0.5,1.0,6.94,1.0,56.99,2.5,1.75,21.866667,24.695122,22,1,0,false,2.302585,4.144444,1.705793
2,2024-01-17 12:59:13,2024-01-17 13:27:08,1,5.37,1,"""N""",236,158,1,29.6,0.0,0.5,3.0,0.0,1.0,36.6,2.5,0.0,27.916667,11.541493,12,3,2,false,1.851599,5.512104,1.060299


In [17]:
df = pl.read_csv(raw_dir/'taxi_zone_lookup.csv')
df.head()

LocationID,Borough,Zone,service_zone
i64,str,str,str
1,"""EWR""","""Newark Airport""","""EWR"""
2,"""Queens""","""Jamaica Bay""","""Boro Zone"""
3,"""Bronx""","""Allerton/Pelham Gardens""","""Boro Zone"""
4,"""Manhattan""","""Alphabet City""","""Yellow Zone"""
5,"""Staten Island""","""Arden Heights""","""Boro Zone"""


In [18]:
#Zone features

# read zone table
zone_lookup = pl.read_csv(raw_dir/'taxi_zone_lookup.csv')

#join the zone lookup table to enriched table
enriched = enriched.join(
    zone_lookup.select([
        pl.col("LocationID"),
        pl.col("Borough")
    ]),
    left_on='PULocationID',
    right_on='LocationID',
    how='left'
).rename({'Borough': 'pickup_borough'})

enriched = enriched.join(
    zone_lookup.select([
        pl.col("LocationID"),
        pl.col("Borough")
    ]),
    left_on='DOLocationID',
    right_on='LocationID',
    how='left'
).rename({'Borough': 'dropoff_borough'})

enriched.sample(n=6)

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,trip_duration_minutes,trip_speed_mph,pickup_hour,pickup_weekday,pickup_day_of_week,is_weekend,log_trip_distance,fare_per_mile,fare_per_minute,pickup_borough,dropoff_borough
i32,datetime[ns],datetime[ns],i64,f64,i64,str,i32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8,i8,i8,bool,f64,f64,f64,str,str
2,2024-01-13 18:37:58,2024-01-13 19:09:05,1,6.88,1,"""N""",239,232,1,35.9,0.0,0.5,11.97,0.0,1.0,51.87,2.5,0.0,31.116667,13.266202,18,6,5,true,2.064328,5.218023,1.153723,"""Manhattan""","""Manhattan"""
2,2024-01-24 10:03:00,2024-01-24 10:35:42,3,1.83,1,"""N""",68,161,1,26.1,0.0,0.5,4.0,0.0,1.0,34.1,2.5,0.0,32.7,3.357798,10,3,2,false,1.040277,14.262295,0.798165,"""Manhattan""","""Manhattan"""
2,2024-01-19 19:40:09,2024-01-19 19:43:49,1,0.74,1,"""N""",239,142,1,5.8,2.5,0.5,3.69,0.0,1.0,15.99,2.5,0.0,3.666667,12.109091,19,5,4,false,0.553885,7.837838,1.581818,"""Manhattan""","""Manhattan"""
2,2024-01-30 08:43:49,2024-01-30 08:59:17,1,1.49,1,"""N""",170,163,1,14.2,0.0,0.5,2.0,0.0,1.0,20.2,2.5,0.0,15.466667,5.780172,8,2,1,false,0.912283,9.530201,0.918103,"""Manhattan""","""Manhattan"""
2,2024-01-04 14:32:30,2024-01-04 14:42:19,5,1.24,1,"""N""",246,48,1,10.7,0.0,0.5,3.68,0.0,1.0,18.38,2.5,0.0,9.816667,7.578947,14,4,3,false,0.806476,8.629032,1.089983,"""Manhattan""","""Manhattan"""
1,2024-01-05 16:55:18,2024-01-05 16:59:47,1,0.9,1,"""N""",263,141,1,7.2,5.0,0.5,2.7,0.0,1.0,16.4,2.5,0.0,4.483333,12.04461,16,5,4,false,0.641854,8.0,1.605948,"""Manhattan""","""Manhattan"""


In [19]:
# convert from polars to pandas
import pandas as pd

enriched_pd = enriched.to_pandas()

enriched_pd.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,trip_speed_mph,pickup_hour,pickup_weekday,pickup_day_of_week,is_weekend,log_trip_distance,fare_per_mile,fare_per_minute,pickup_borough,dropoff_borough
0,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1,1.8,1,N,140,236,1,...,16.363636,0,1,0,False,1.029619,5.555556,1.515152,Manhattan,Manhattan
1,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1,4.7,1,N,236,79,1,...,15.739535,0,1,0,False,1.740466,4.957447,1.300465,Manhattan,Manhattan
2,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1,1.4,1,N,79,211,1,...,10.120482,0,1,0,False,0.875469,7.142857,1.204819,Manhattan,Manhattan
3,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1,0.8,1,N,211,148,1,...,7.868852,0,1,0,False,0.587787,9.875000,1.295082,Manhattan,Manhattan
4,1,2024-01-01 00:54:08,2024-01-01 01:26:31,1,4.7,1,N,148,141,1,...,8.708183,0,1,0,False,1.740466,6.297872,0.914050,Manhattan,Manhattan


In [20]:
from sklearn.preprocessing import OneHotEncoder
#ensure a regular dense array is returned and sklearn do not crash with unknown category
ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")

#look at columns and learn the categories
encoded_2d_array = ohe.fit_transform(enriched_pd[["pickup_borough", "dropoff_borough"]])

# to avoid the new columns being numbered 0,1,2...
encoded_cols = ohe.get_feature_names_out(["pickup_borough", "dropoff_borough"])

#turn encoded output to pandas table and add it to the enriched_pd dataframe
encoded_df = pd.DataFrame(encoded_2d_array, columns=encoded_cols, index=enriched_pd.index)
enriched_pd = pd.concat([enriched_pd, encoded_df], axis=1)

enriched_pd.sample(n=5)

ModuleNotFoundError: No module named 'sklearn'

### 2. Target Variable Creation

In [ ]:
# tip_amount (continuous, for regression)
y_reg = enriched_pd["tip_amount"]

#high_tip (binary: 1 if tip_amount > 20% of fare_amount, 0 otherwise, for classification
enriched_pd["high_tip"] = (
    (enriched_pd["tip_amount"] > 0) &
    (enriched_pd["tip_amount"] > 0.2 * enriched_pd["fare_amount"])
).astype(int)

y_clf = enriched_pd["high_tip"]

enriched_pd.sample(n=5)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,pickup_borough_Unknown,dropoff_borough_Bronx,dropoff_borough_Brooklyn,dropoff_borough_EWR,dropoff_borough_Manhattan,dropoff_borough_N/A,dropoff_borough_Queens,dropoff_borough_Staten Island,dropoff_borough_Unknown,high_tip
2233423,2,2024-01-31 10:47:17,2024-01-31 11:30:01,1,8.49,1,N,138,164,1,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0
1549214,1,2024-01-22 17:05:27,2024-01-22 17:18:30,1,1.60,1,N,234,79,1,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1
369444,2,2024-01-06 16:21:28,2024-01-06 16:43:48,2,3.34,1,N,113,237,1,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0
1454189,2,2024-01-21 04:24:21,2024-01-21 04:29:14,1,1.93,1,N,233,79,1,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0
1496001,2,2024-01-21 19:12:22,2024-01-21 19:33:24,1,9.41,1,N,138,237,1,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1


In [ ]:

enriched.select(["tip_amount", "fare_amount"]).sample(n=5)

tip_amount,fare_amount
f64,f64
19.28,85.94
2.94,10.7
8.95,33.8
2.58,10.7
2.52,12.8


### 3. Data Splitting & Scaling

In [ ]:
# Split data into training (70%), validation (15%), and test (15%) sets using stratified sampling for the classification target
from sklearn.model_selection import train_test_split

X= enriched_pd.drop(["tip_amount", "high_tip"], axis=1)
y_reg = enriched_pd["tip_amount"]
y_clf = enriched_pd["high_tip"]

X_train, X_temp, y_train_reg, y_temp_reg, y_train, y_temp = train_test_split(
    X, y_reg, y_clf, test_size=0.3, random_state=42, stratify=y_clf
)

X_val, X_test, y_val_reg, y_test_reg, y_val, y_test = train_test_split(
    X_temp, y_temp_reg, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

In [ ]:
#identify column types
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "bool"]).columns.tolist()

print(f'Numeric features: {len(numeric_features)}')
print(f'Categorical features: {len(categorical_features)}')

Numeric features: 33
Categorical features: 4


In [ ]:
# Apply StandardScaler to numeric features; fit on training data only
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

#numeric pipeline: impute missing values then scale
numeric_transformer = Pipeline(steps=[
  ('imputer', SimpleImputer(strategy='median')),
  ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(
transformers=[
  ('num', numeric_transformer, numeric_features),
])

X_train_scaled = preprocessor.fit_transform(X_train)
X_val_scaled = preprocessor.transform(X_val)
X_test_scaled = preprocessor.transform(X_test)


In [ ]:
# included_features = numeric_features
# excluded_by_preprocessor = [col for col in X.columns if col not in included_features]

# print("Included in preprocessing:")
# print(included_features)

# print("\nExcluded by preprocessing:")
# print(excluded_by_preprocessor)

In [ ]:
#Document the number of samples in each split and the class distribution of high_tip in each split
print("Number of samples in each split:")
print("Training set size:", len(X_train))
print("Validation set size:", len(X_val))
print("Test set size:", len(X_test))

print("\nClass distribution of high_tip in Training set:")
print(y_train.value_counts())
print(y_train.value_counts(normalize=True))

print("\nClass distribution of high_tip in Validation set:")
print(y_val.value_counts())
print(y_val.value_counts(normalize=True))

print("\nClass distribution of high_tip in Test set:")
print(y_test.value_counts())
print(y_test.value_counts(normalize=True))

Number of samples in each split:
Training set size: 1608866
Validation set size: 344757
Test set size: 344757

Class distribution of high_tip in Training set:
high_tip
1    1221650
0     387216
Name: count, dtype: int64
high_tip
1    0.759324
0    0.240676
Name: proportion, dtype: float64

Class distribution of high_tip in Validation set:
high_tip
1    261782
0     82975
Name: count, dtype: int64
high_tip
1    0.759323
0    0.240677
Name: proportion, dtype: float64

Class distribution of high_tip in Test set:
high_tip
1    261782
0     82975
Name: count, dtype: int64
high_tip
1    0.759323
0    0.240677
Name: proportion, dtype: float64


In [ ]:
# print(enriched_pd["high_tip"].value_counts())
# print(enriched_pd["high_tip"].value_counts(normalize=True))

In [ ]:
#Print a summary of feature names, types, and any features excluded from modeling (and why)
print("Feature names used for modeling:")
print(X.columns.tolist())

print("\nFeature data types:")
print(X.dtypes)

print("\nCount of each data type:")
print(X.dtypes.value_counts())

Feature names used for modeling:
['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'trip_duration_minutes', 'trip_speed_mph', 'pickup_hour', 'pickup_weekday', 'pickup_day_of_week', 'is_weekend', 'log_trip_distance', 'fare_per_mile', 'fare_per_minute', 'pickup_borough', 'dropoff_borough', 'pickup_borough_Bronx', 'pickup_borough_Brooklyn', 'pickup_borough_EWR', 'pickup_borough_Manhattan', 'pickup_borough_N/A', 'pickup_borough_Queens', 'pickup_borough_Staten Island', 'pickup_borough_Unknown', 'dropoff_borough_Bronx', 'dropoff_borough_Brooklyn', 'dropoff_borough_EWR', 'dropoff_borough_Manhattan', 'dropoff_borough_N/A', 'dropoff_borough_Queens', 'dropoff_borough_Staten Island', 'dropoff_borough_Unknown']

Feature data types:
VendorI

In [ ]:
print(f"\nNumeric features ({len(numeric_features)}):")
print(numeric_features)

print(f"\nCategorical features ({len(categorical_features)}):")
print(categorical_features)


Numeric features (33):
['passenger_count', 'trip_distance', 'RatecodeID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'trip_duration_minutes', 'trip_speed_mph', 'log_trip_distance', 'fare_per_mile', 'fare_per_minute', 'pickup_borough_Bronx', 'pickup_borough_Brooklyn', 'pickup_borough_EWR', 'pickup_borough_Manhattan', 'pickup_borough_N/A', 'pickup_borough_Queens', 'pickup_borough_Staten Island', 'pickup_borough_Unknown', 'dropoff_borough_Bronx', 'dropoff_borough_Brooklyn', 'dropoff_borough_EWR', 'dropoff_borough_Manhattan', 'dropoff_borough_N/A', 'dropoff_borough_Queens', 'dropoff_borough_Staten Island', 'dropoff_borough_Unknown']

Categorical features (4):
['store_and_fwd_flag', 'is_weekend', 'pickup_borough', 'dropoff_borough']


In [ ]:
print("\nExcluded features:")
print("\ntip_amount was excluded because it is a regression target and would cause target leakage")
print("\nhigh_tip was excluded because it is a classification target and would cause target leakage")
print("\nVendorID was excluded it is an identifier and not included in the numeric preprocessing pipepline")
print("\ntpep_pickup_datetime was excluded because it is a date and not included in the numeric preprocessing pipepline")
print("\ntpep_dropoff_datetime was excluded because it is a date and not included in the numeric preprocessing pipepline")
print("\nstore_and_fwd_flag, is_weekend, pickup_borough and dropoff_borough was excluded because it is categorical")
print("\nPULocationID was excluded because it is a identifier")
print("\nDOLocationID was excluded because it is a identifier")
print("\npickup_hour, pickup_weekday and pickup_day_of_week was excluded because the data type is int8 and only data type int64 and float64 was included in the pipeline")


Excluded features:

tip_amount was excluded because it is a regression target and would cause target leakage

high_tip was excluded because it is a classification target and would cause target leakage

VendorID was excluded it is an identifier and not included in the numeric preprocessing pipepline

tpep_pickup_datetime was excluded because it is a date and not included in the numeric preprocessing pipepline

tpep_dropoff_datetime was excluded because it is a date and not included in the numeric preprocessing pipepline

store_and_fwd_flag, is_weekend, pickup_borough and dropoff_borough was excluded because it is categorical

PULocationID was excluded because it is a identifier

DOLocationID was excluded because it is a identifier

pickup_hour, pickup_weekday and pickup_day_of_week was excluded because the data type is int8 and only data type int64 and float64 was included in the pipeline


## Part 2: Model Training & Tuning

### Baseline Models

In [ ]:
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)
import numpy as np
import pandas as pd

In [ ]:
#define regression models
reg_models = {
    "Linear Regression": LinearRegression(),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=100, random_state=42)
   }


In [ ]:
reg_results = {}

for name, reg_model in reg_models.items():
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("regressor", reg_model)
    ])

    pipe.fit(X_train, y_train_reg)
    y_val_pred = pipe.predict(X_val)

    reg_results[name] = {
        "MAE": mean_absolute_error(y_val_reg, y_val_pred),
        "RMSE": np.sqrt(mean_squared_error(y_val_reg, y_val_pred)),
        "R2": r2_score(y_val_reg, y_val_pred)
    }

In [ ]:
print("Regression Model Performance:")
for name, metrics in reg_results.items():
    print(f"{name}:")
    for metric_name, metric_value in metrics.items():
        print(f"  {metric_name}: {metric_value}")
    print()

Regression Model Performance:
Linear Regression:
  MAE: 0.5444301490712642
  RMSE: 0.7508882511918472
  R2: 0.9615517749941261

Random Forest Regressor:
  MAE: 0.05391087009618029
  RMSE: 0.5401749327943285
  R2: 0.9801026825077261



In [ ]:
# define classification models
clf_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest Classifier": RandomForestClassifier(n_estimators=100, random_state=42)
}

In [ ]:

clf_results = {}

for name, clf_model in clf_models.items():
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", clf_model)
    ])

    pipe.fit(X_train, y_train)
    y_val_pred = pipe.predict(X_val)
    y_val_prob = pipe.predict_proba(X_val)[:, 1]

    clf_results[name] = {
        "Accuracy": accuracy_score(y_val, y_val_pred),
        "Precision": precision_score(y_val, y_val_pred),
        "Recall": recall_score(y_val, y_val_pred),
        "F1": f1_score(y_val, y_val_pred),
        "AUC-ROC": roc_auc_score(y_val, y_val_prob)
    }

In [ ]:
print("Classification Model Performance:")
for name, metrics in clf_results.items():
    print(f"{name}:")
    for metric_name, metric_value in metrics.items():
        print(f"  {metric_name}: {metric_value}")
    print()

### 5. Hyperparameter Tuning

In [ ]:
#Using RandomizedSearchCV with 3 hyperparameters
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

In [ ]:
# Create pipeline with Random Forest
rf_pipeline = Pipeline(steps=[
  ('preprocessor', preprocessor),
  ('classifier', RandomForestClassifier(random_state=42))
])

param_distributions = {
  'classifier__n_estimators': randint(50, 200),
  'classifier__max_depth': [5, 10, 20, None],
  'classifier__min_samples_split': randint(2, 10),
  # 'classifier__min_samples_leaf': randint(1, 5),
}

random_search = RandomizedSearchCV(
  rf_pipeline, param_distributions, n_iter=20, cv=5,
  scoring="f1", n_jobs=-1, random_state=42
)

X_train_tune, _, y_train_tune, _ = train_test_split(X_train, y_train, train_size=200000, random_state=42, stratify=y_train)

# X_train_tune = X_train.sample(200000, random_state=42)
# y_train_tune = y_train.loc[X_train_tune.index]

random_search.fit(X_train_tune, y_train_tune)

# # took 16 mins

In [ ]:
# # Create pipeline with Random Forest
# rf_pipeline = Pipeline(steps=[
#   ('preprocessor', preprocessor),
#   ('classifier', RandomForestClassifier(random_state=42))
# ])

# param_distributions = {
#   'classifier__n_estimators': randint(50, 300),
#   'classifier__max_depth': [5, 10, 20, None],
#   'classifier__min_samples_split': randint(2, 10),
#   # 'classifier__min_samples_leaf': randint(1, 5),
# }

# random_search = RandomizedSearchCV(
#   rf_pipeline, param_distributions, n_iter=10, cv=5,
#   scoring="f1", n_jobs=-1, random_state=42
# )

# X_train_tune = X_train.sample(200000, random_state=42)
# y_train_tune = y_train.loc[X_train_tune.index]

# random_search.fit(X_train_tune, y_train_tune)
# #n iter at 20 w leaf commented took over 20 mins


In [ ]:
print("Hyperparameter search space:")
print(param_distributions)

print("\nBest parameters found:")
print(random_search.best_params_)

## Neural Network

In [ ]:
#classification task

import torch.nn as nn

# Move tensor to GPU (if available)
if torch.cuda.is_available():
x_gpu = x.to('cuda')
print(f'Tensor on: {x_gpu.device}')

# Operations on GPU tensors stay on GPU
y_gpu = y.to('cuda')
z_gpu = x_gpu + y_gpu
# Move back to CPU for printing/NumPy
z_cpu = z_gpu.cpu()
print(f'Result: {z_cpu}')


In [ ]:
#Convert to PyTorch tensors
X_train_scaled = preprocessor.fit_transform(X_train)
X_val_scaled = preprocessor.transform(X_val)
X_test_scaled = preprocessor.transform(X_test)

X_train_tensor = torch.FloatTensor(X_train_scaled).to(device)
y_train_tensor = torch.FloatTensor(y_train.values).to(device)

X_val_tensor = torch.FloatTensor(X_val_scaled).to(device)
y_val_tensor = torch.FloatTensor(y_val.values).to(device)

X_test_tensor = torch.FloatTensor(X_test_scaled).to(device)
y_test_tensor = torch.FloatTensor(y_test.values).to(device)

print(f'Training set: {X_train_tensor.shape}')
print(f'Validation set: {X_val_tensor.shape}')
print(f'Test set: {X_test_tensor.shape}')
print(f'Input features': {X_train_tensor.shape[1]})

In [ ]:
import torch.nn as nn

class TipClassifier(nn.Module):
  def __init__(self, input_size, hidden_sizes=[128, 64], dropout_rate=0.3):
    super(TipClassifier, self).__init__()

    #building layers dynamically
    layers = []
    prev_size = input_size

    for hidden_size in hidden_sizes:
      layers.append(nn.Linear(prev_size, hidden_size))
      layers.append(nn.ReLU())
      layers.append(nn.Dropout(dropout_rate))
      prev_size = hidden_size

    layers.append(nn.Linear(prev_size, 1))
    layers.append(nn.Sigmoid())

    self.model = nn.Sequential(*layers)

  def forward(self, x):
    return self.network(x).squeeze()

In [ ]:
#instantiate model and inspect
input_size = X_train_tensor.shape[1]
model = TipClassifier(input_size, hidden_sizes=[128, 64]).to(device)

print(model)
print(f'\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Trainable parameters: {sum(p.numel() for p in model.parameters() if
p.requires_grad):,}')

AI USAGE
- Used chat gpt to look into log transformed distance and what method to use.
  - Learned that this is done since usually the distance is skewed to the right so log transform comes in to reduce the effect of the extremely large values.
  - Chose log1p() or log() since it handles 0 values safely.
- Used chatgpt to understand when to use one hot encoding and when to use label encoding.
  - While label encoding would have less columns compared to one hot, I decided to go with one hot encoding since it will work across all the models for this assignment. (specifically logistic and linear regression model and neural network. These models would have the risk of misreading label encoded categories.)
  - How to do one hot encoding
  